# Why do I keep losing? Draft-only Dota 2 win prediction

This notebook walks through training a model that predicts Dota 2 match outcomes
using *only* the draft (which heroes are picked, nothing else). The goal was to
test the "we already lost in the draft" myth, and to see whether representing
hero picks as an **image of hero icons** lets a vision model find patterns that
plain hero-id features can't.

Data: 5,939 professional matches, all on the same patch (to remove patch-to-patch
balance shifts from the picture), pulled from the OpenDota API. Each match becomes
a 2x5 grid image, Radiant on top, Dire on bottom, heroes ordered by role.

## 1. Setup: mount drive and load the project

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Persistent data
DATA_DIR = Path(
    "/content/drive/MyDrive/Github/why_do_I_keep_losing/data"
)

print(DATA_DIR)


In [ ]:
# Sanity check: confirm the drive mounted and hero icons are reachable
from pathlib import Path
from IPython.display import display
from PIL import Image

image_path = DATA_DIR / "heroes" / "icons" / "anti_mage.png"
image = Image.open(image_path)

print(image_path)
print("Exists:", image_path.exists())
print("Size:", image.size)
print("Mode:", image.mode)
display(image)


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Zekesy/why-do-I-keep-losing.git"
REPO_DIR = Path("/content/why_do_I_keep_losing")
SRC_DIR = REPO_DIR / "src"

if not REPO_DIR.exists():
    print("[INFO] Project not found. Cloning from GitHub...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("[INFO] Project already exists.")
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=False)
    print("[INFO] Pulling latest changes...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for module_name in list(sys.modules):
    if module_name == "why_do_I_keep_losing" or module_name.startswith("why_do_I_keep_losing."):
        del sys.modules[module_name]

import why_do_I_keep_losing
print("Package imported from:", why_do_I_keep_losing.__file__)

from why_do_I_keep_losing.utils.data_setup import create_dataloaders
print("data_setup imported successfully")

os.chdir(REPO_DIR)
print("Project ready at", REPO_DIR)


## 2. Building the dataset, and hurdle #1: the model that only ever guessed

Every match becomes a 2x5 hero icon grid. First attempt at this pipeline resized
a full grid of 224x224 icons down into a single 224x224 canvas at the end, which
crushed every hero portrait into an unrecognizable blurry smear. Test accuracy sat
at exactly the class-prior number (0.5285 / 0.4715) every single epoch, no matter
the architecture, which is the signature of a model that's just guessing the
majority class every time.

The fix: resize each icon to a small, patch-aligned cell size *before* assembling
the grid, so nothing gets squashed at the end, and make sure normalization matches
whatever pretrained backbone is being used (ImageNet mean/std vs a flat 0.5/0.5/0.5,
depending on the checkpoint).

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from why_do_I_keep_losing.utils.data_setup import create_dataloaders
from why_do_I_keep_losing.models.hero_pic_ViT import HeroPicViT
from why_do_I_keep_losing.models.hero_pic_dataset import DotaHeroPicViTDataset
from timm.data import resolve_data_config
from torchvision import transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

PARQUET_PATH = DATA_DIR / "processed"
ICONS_DIR = DATA_DIR / "heroes" / "icons"

# Full-canvas layout: icon cells sized to fill the 224x224 canvas with no
# empty padding, instead of a small patch-aligned cell centered on gray space.
CELL_W = 224 // 5   # = 44
CELL_H = 224 // 2   # = 112

model = HeroPicViT(model_name="vit_base_patch16_224", pretrained=True, drop_rate=0.2)
model.to(device)

data_config = resolve_data_config({}, model=model.backbone)
vit_transforms = transforms.Compose([
    transforms.Resize((CELL_H, CELL_W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=data_config["mean"], std=data_config["std"]),
])

print(f"Using mean={data_config['mean']}, std={data_config['std']}")


In [ ]:
# Visual check: confirm hero icons are crisp and distinguishable, not blurry
train_loader, val_loader, test_loader, *_ = create_dataloaders(
    parquet_dir=str(PARQUET_PATH),
    icons_dir=str(ICONS_DIR),
    transform=vit_transforms,
    random_order=False,
    batch_size=8,
    val_split=0.15,
    test_split=0.15,
    num_workers=0,
)
X, y = next(iter(train_loader))
print(f"Batch shape: {X.shape}  (expect [8, 3, 224, 224])")
print(f"min/mean/max: {X.min().item():.3f} / {X.mean().item():.3f} / {X.max().item():.3f}")
print(f"labels: {y.tolist()}")

def denorm(t, mean, std):
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, ax in enumerate(axes):
    img = denorm(X[i], data_config["mean"], data_config["std"])
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(f"label={y[i].item()}")
    ax.axis("off")
plt.suptitle("Visual check: should clearly see 10 distinct hero icons (2x5 grid)")
plt.show()


## 3. Auditing the pipeline before trusting any result

Before trusting model results, worth ruling out bugs: mismatched labels, hero
overlap between teams, leakage between train/val/test, and confirming the "single
patch" claim actually holds in the data. Everything below came back clean, which
meant the weirdness in early model results was a real modeling problem, not a
pipeline bug.

In [ ]:
import pandas as pd

dataset = DotaHeroPicViTDataset(
    parquet_dir=str(PARQUET_PATH),
    icons_dir=str(ICONS_DIR),
    transform=None,
    random_order=False,
)
df = dataset.df

# ---- Patch distribution: confirm single-patch claim ----
print("[1] Patch value counts:")
print(df["patch"].value_counts(dropna=False).sort_index())
print(f"Number of distinct patches: {df['patch'].nunique(dropna=False)}")
print()

# ---- Label balance ----
print("[2] Label balance:")
print(df["winning_team"].value_counts(normalize=True))
print()

# ---- Degenerate picks / hero overlap between teams ----
print("[3] Checking for degenerate picks (overlap or <5 heroes per team):")
bad_rows = 0
for _, row in df.iterrows():
    r_ids = set(h["hero_id"] for h in row["radiant_heroes"])
    d_ids = set(h["hero_id"] for h in row["dire_heroes"])
    if len(r_ids) < 5 or len(d_ids) < 5:
        bad_rows += 1
    if r_ids & d_ids:
        bad_rows += 1
        print(f"  overlap in match_id={row['match_id']}: {r_ids & d_ids}")
print(f"-> {bad_rows} problematic rows out of {len(df)}")
print()

# ---- __getitem__ label matches dataframe label ----
print("[4] Verifying dataset labels match dataframe labels (first 10 rows):")
mismatches = 0
for idx in range(10):
    row = df.iloc[idx]
    _, label = dataset[idx]
    expected = 1.0 if row["winning_team"] == "radiant" else 0.0
    if label.item() != expected:
        mismatches += 1
print(f"-> {mismatches} mismatches")
print()

# ---- Leakage between splits ----
print("[5] Checking for match_id leakage across splits:")
_train_loader, _val_loader, _test_loader, *_ = create_dataloaders(
    parquet_dir=str(PARQUET_PATH),
    icons_dir=str(ICONS_DIR),
    transform=vit_transforms,
    random_order=False,
    batch_size=32,
    val_split=0.15,
    test_split=0.15,
    num_workers=0,
)

def get_match_ids_from_loader(loader):
    ids = set()
    ds = loader.dataset
    indices = ds.indices if hasattr(ds, "indices") else range(len(ds))
    base_df = ds.dataset.df if hasattr(ds, "dataset") else ds.df
    for i in indices:
        ids.add(base_df.iloc[i]["match_id"])
    return ids

train_ids = get_match_ids_from_loader(_train_loader)
val_ids = get_match_ids_from_loader(_val_loader)
test_ids = get_match_ids_from_loader(_test_loader)
print("train/val overlap:", len(train_ids & val_ids))
print("train/test overlap:", len(train_ids & test_ids))
print("val/test overlap:", len(val_ids & test_ids))


## 4. Training utilities

Core training loop with early stopping (restores the best validation-loss
checkpoint rather than whatever the last epoch happened to land on, since the
first version of this loop kept training well past the point of overfitting),
plus a `RunConfig` dataclass and `run_experiment` runner used for the full
architecture sweep below.

In [ ]:
import json
import time
import traceback
from pathlib import Path
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
from tqdm import tqdm

from why_do_I_keep_losing.utils.data_setup import create_dataloaders
from why_do_I_keep_losing.engine.train import train_step, eval_step
from why_do_I_keep_losing.models.hero_pic_ViT import HeroPicViT
from timm.data import resolve_data_config
from torchvision import transforms

RUNS_ROOT = Path("/content/drive/MyDrive/why_do_I_keep_losing/runs")
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = DATA_DIR / "processed"
ICONS_DIR = DATA_DIR / "heroes" / "icons"


def train(
    model, train_dataloader, test_dataloader, optimizer, loss_fn,
    epochs, device, patience: int = 3,
):
    """Trains with early stopping on validation loss, restoring the best
    checkpoint at the end instead of whatever the last epoch produced."""
    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    model.to(device)

    best_test_loss = float("inf")
    epochs_without_improvement = 0
    best_state = None

    for epoch in tqdm(range(epochs), desc="Epochs"):
        train_loss, train_acc = train_step(model, train_dataloader, loss_fn, optimizer, device)
        test_loss, test_acc = eval_step(model, test_dataloader, loss_fn, device)

        print(
            f"Epoch: {epoch + 1:02d} | train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | test_loss: {test_loss:.4f} | test_acc: {test_acc:.4f}"
        )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)

        if test_loss < best_test_loss:
            best_test_loss = test_loss
            epochs_without_improvement = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"[EarlyStopping] No improvement for {patience} epochs. Stopping at epoch {epoch+1}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return results


@dataclass
class RunConfig:
    name: str
    model_name: str = "vit_base_patch16_224"
    pretrained: bool = True
    drop_rate: float = 0.2
    random_order: bool = False
    batch_size: int = 32
    lr: float = 1e-4
    freeze_backbone: bool = True
    weight_decay: float = 1e-4
    epochs: int = 3
    seed: int = 42
    patience: int = 3
    save_weights: bool = False


def run_experiment(cfg: RunConfig):
    run_dir = RUNS_ROOT / cfg.name
    run_dir.mkdir(parents=True, exist_ok=True)

    with open(run_dir / "config.json", "w") as f:
        json.dump(asdict(cfg), f, indent=2)

    torch.manual_seed(cfg.seed)

    model = HeroPicViT(model_name=cfg.model_name, pretrained=cfg.pretrained, drop_rate=cfg.drop_rate)

    if cfg.freeze_backbone:
        for param in model.backbone.parameters():
            param.requires_grad = False

    data_config = resolve_data_config({}, model=model.backbone)

    # Full-canvas icon cells (no wasted padding), fixes the original
    # blurry-icon bug from squashing a big grid down to 224x224 at the end.
    CELL_W = 224 // 5
    CELL_H = 224 // 2
    vit_transforms = transforms.Compose([
        transforms.Resize((CELL_H, CELL_W)),
        transforms.ToTensor(),
        transforms.Normalize(mean=data_config["mean"], std=data_config["std"]),
    ])

    train_loader, val_loader, test_loader, *_ = create_dataloaders(
        parquet_dir=str(PARQUET_PATH),
        icons_dir=str(ICONS_DIR),
        transform=vit_transforms,
        random_order=cfg.random_order,
        batch_size=cfg.batch_size,
        val_split=0.15,
        test_split=0.15,
        num_workers=0,
    )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    start = time.time()
    results = train(
        model=model, train_dataloader=train_loader, test_dataloader=val_loader,
        optimizer=optimizer, loss_fn=loss_fn, epochs=cfg.epochs, device=device,
        patience=cfg.patience,
    )
    elapsed = time.time() - start

    if cfg.save_weights:
        torch.save(model.state_dict(), run_dir / "model_weights.pt")

    def clean(obj):
        if isinstance(obj, torch.Tensor):
            return obj.item() if obj.numel() == 1 else obj.tolist()
        if isinstance(obj, dict):
            return {k: clean(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [clean(v) for v in obj]
        return obj

    results_clean = clean(results)
    results_clean["_elapsed_seconds"] = elapsed

    with open(run_dir / "results.json", "w") as f:
        json.dump(results_clean, f, indent=2)

    print(f"Finished {cfg.name} in {elapsed/60:.1f} min -> saved to {run_dir}")
    return results_clean


## 5. Hurdle #2: chasing the vision approach across architectures

Ran a wide sweep across ViT-Base/Small/Tiny, DeiT3, Swin-Tiny, and ResNet18, at
various sizes, patch sizes, frozen vs fully fine-tuned backbones, pretrained vs
trained from scratch, and both hero role ordering and randomized ordering as an
ablation.

Result across every single one of these: test accuracy stuck around 49-53%,
indistinguishable from guessing. The failure mode shifted along the way, though.
Early on, train accuracy was *also* stuck near chance (the model couldn't even fit
the training set). After the resize fix, fine-tuned ResNet/ViT runs showed classic
overfitting instead, train accuracy climbing into the 70-90% range while test
accuracy stayed flat and test loss started rising. Early stopping plus dropout and
weight decay fixed the overfitting shape, but the test ceiling itself never moved.

In [ ]:
from torchvision import transforms

configs = [

    # ================================================================
    # 1. ViT BASELINE
    #    Compare hero-role ordering and training duration.
    # ================================================================
    RunConfig(name="vit_base_roles", model_name="vit_base_patch16_224", random_order=False),
    RunConfig(name="vit_base_random_order", model_name="vit_base_patch16_224", random_order=True),
    RunConfig(name="vit_base_roles_epochs_10", model_name="vit_base_patch16_224", random_order=False, epochs=10),
    RunConfig(name="vit_base_random_order_epochs_10", model_name="vit_base_patch16_224", random_order=True, epochs=10),

    # ================================================================
    # 2. VIT MODEL SIZE
    #    Test whether model capacity affects performance.
    # ================================================================
    RunConfig(name="vit_small_roles", model_name="vit_small_patch16_224", random_order=False, drop_rate=0.1),
    RunConfig(name="vit_small_random_order", model_name="vit_small_patch16_224", random_order=True, drop_rate=0.1),
    RunConfig(name="vit_tiny_roles", model_name="vit_tiny_patch16_224", random_order=False, drop_rate=0.1),
    RunConfig(name="vit_tiny_random_order", model_name="vit_tiny_patch16_224", random_order=True, drop_rate=0.1),

    # ================================================================
    # 3. PRETRAINING ABLATION
    #    Does ImageNet pretraining help on the hero-grid task?
    # ================================================================
    RunConfig(name="vit_base_roles_scratch", model_name="vit_base_patch16_224", pretrained=False, random_order=False),
    RunConfig(name="vit_base_random_order_scratch", model_name="vit_base_patch16_224", pretrained=False, random_order=True),

    # ================================================================
    # 4. PATCH SIZE ABLATION
    #    Compare fine-grained 16x16 patches against coarser 32x32
    #    patches while keeping the ViT-Base architecture.
    # ================================================================
    RunConfig(name="vit_base32_roles", model_name="vit_base_patch32_224", random_order=False),
    RunConfig(name="vit_base32_random_order", model_name="vit_base_patch32_224", random_order=True),
    RunConfig(name="vit_base32_roles_scratch", model_name="vit_base_patch32_224", pretrained=False,
              random_order=False, freeze_backbone=False, lr=1e-4, epochs=10),
    RunConfig(name="vit_base32_random_order_scratch", model_name="vit_base_patch32_224", pretrained=False,
              random_order=True, freeze_backbone=False, lr=1e-4, epochs=10),

    # ================================================================
    # 5. ALTERNATIVE TRANSFORMER ARCHITECTURES
    #    Test whether the result generalizes beyond standard ViTs.
    # ================================================================
    RunConfig(name="deit3_small_roles", model_name="deit3_small_patch16_224", random_order=False, drop_rate=0.1),
    RunConfig(name="deit3_small_random_order", model_name="deit3_small_patch16_224", random_order=True, drop_rate=0.1),
    RunConfig(name="swin_tiny_roles", model_name="swin_tiny_patch4_window7_224", random_order=False, drop_rate=0.1),
    RunConfig(name="swin_tiny_random_order", model_name="swin_tiny_patch4_window7_224", random_order=True, drop_rate=0.1),

    # ================================================================
    # 6. RESNET18 - PRETRAINING STRATEGY
    #    Establish a CNN baseline and compare frozen vs fine-tuned
    #    ImageNet features.
    # ================================================================
    RunConfig(name="resnet18_roles_pretrained_frozen", model_name="resnet18", pretrained=True,
              random_order=False, drop_rate=0.1, freeze_backbone=True),
    RunConfig(name="resnet18_random_order_pretrained_frozen", model_name="resnet18", pretrained=True,
              random_order=True, drop_rate=0.1, freeze_backbone=True),
    RunConfig(name="resnet18_roles_pretrained_finetune", model_name="resnet18", pretrained=True,
              random_order=False, drop_rate=0.1, freeze_backbone=False, lr=2e-5, epochs=10),
    RunConfig(name="resnet18_random_order_pretrained_finetune", model_name="resnet18", pretrained=True,
              random_order=True, drop_rate=0.1, freeze_backbone=False, lr=2e-5, epochs=10),
    RunConfig(name="resnet18_roles_scratch", model_name="resnet18", pretrained=False,
              random_order=False, drop_rate=0.1, freeze_backbone=False, lr=1e-4, epochs=10),
    RunConfig(name="resnet18_random_order_scratch", model_name="resnet18", pretrained=False,
              random_order=True, drop_rate=0.1, freeze_backbone=False, lr=1e-4, epochs=10),

    # ================================================================
    # 7. RESNET18 - REGULARIZATION ABLATION
    #    Starting from the pretrained fine-tuning baseline, test
    #    whether stronger regularization improves generalization.
    #    Baseline: dropout=0.1, weight_decay=1e-4, patience=3
    # ================================================================
    RunConfig(name="resnet18_roles_finetune_earlystop", model_name="resnet18", pretrained=True,
              random_order=False, freeze_backbone=False, drop_rate=0.1,
              lr=2e-5, weight_decay=1e-4, epochs=15, patience=3),
    RunConfig(name="resnet18_roles_finetune_dropout0.4", model_name="resnet18", pretrained=True,
              random_order=False, freeze_backbone=False, drop_rate=0.4,
              lr=2e-5, weight_decay=1e-4, epochs=15, patience=3),
    RunConfig(name="resnet18_roles_finetune_wd1e-2", model_name="resnet18", pretrained=True,
              random_order=False, freeze_backbone=False, drop_rate=0.2,
              lr=2e-5, weight_decay=1e-2, epochs=15, patience=3),
    RunConfig(name="resnet18_roles_finetune_dropout0.4_wd1e-2", model_name="resnet18", pretrained=True,
              random_order=False, freeze_backbone=False, drop_rate=0.4,
              lr=2e-5, weight_decay=1e-2, epochs=15, patience=3),
    RunConfig(name="resnet18_random_order_finetune_dropout0.4_wd1e-2", model_name="resnet18", pretrained=True,
              random_order=True, freeze_backbone=False, drop_rate=0.4,
              lr=2e-5, weight_decay=1e-2, epochs=15, patience=3),
]

all_results = {}
for cfg in configs:
    print(f"\n{'='*60}\nStarting: {cfg.name}\n{'='*60}")
    try:
        all_results[cfg.name] = run_experiment(cfg)
    except Exception as e:
        err_dir = RUNS_ROOT / cfg.name
        err_dir.mkdir(parents=True, exist_ok=True)
        with open(err_dir / "error.log", "w") as f:
            f.write(traceback.format_exc())
        print(f"XXX {cfg.name} failed: {e}")
        continue

print("\nAll done. Summary:")
for name, res in all_results.items():
    print(name, "->", {k: v for k, v in res.items() if k != "_elapsed_seconds"})


## 6. Hurdle #3: a properly boring baseline

Every vision architecture above was stuck near chance. Before concluding the
*task* was unlearnable, the obvious sanity check is a dead-simple baseline that
has nothing to do with images at all: one-hot encode which heroes are on which
team and fit a Gradient Boosted Trees classifier.

This confirms whether the images even carry a learnable signal in the first
place, independent of any architecture choice.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

# Determine actual max hero_id across the dataset (hero ids aren't contiguous)
max_hero_id = 0
for _, row in dataset.df.iterrows():
    for h in list(row["radiant_heroes"]) + list(row["dire_heroes"]):
        max_hero_id = max(max_hero_id, h["hero_id"])

num_heroes = max_hero_id + 1
print(f"max_hero_id={max_hero_id}, using num_heroes={num_heroes}")

def match_to_vector(row):
    vec = np.zeros(num_heroes * 2)
    for h in row["radiant_heroes"]:
        vec[h["hero_id"]] = 1
    for h in row["dire_heroes"]:
        vec[num_heroes + h["hero_id"]] = 1
    return vec

X = np.stack([match_to_vector(row) for _, row in dataset.df.iterrows()])
y = (dataset.df["winning_team"] == "radiant").astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
probs = clf.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"AUC: {roc_auc_score(y_test, probs):.4f}")


## 7. Hurdle #4: the embedding MLP

The GBM found real (if modest) signal that every vision architecture missed,
around 54.3-54.5% accuracy, 0.56 AUC. That points at the image representation
itself as the bottleneck, not the underlying task.

To confirm, here's the neural network equivalent of the one-hot GBM approach: a
small MLP with a *learned* embedding table per hero id (instead of a fixed
one-hot row), radiant and dire pooled separately and combined before a couple of
dense layers. This lands in almost the same place as the GBM, which is a good
sign, two very different model families independently agreeing on the ceiling.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class HeroPickDataset(Dataset):
    def __init__(self, df, hero_id_to_idx):
        self.df = df.reset_index(drop=True)
        self.hero_id_to_idx = hero_id_to_idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        radiant_ids = torch.tensor(
            [self.hero_id_to_idx[h["hero_id"]] for h in row["radiant_heroes"][:5]],
            dtype=torch.long,
        )
        dire_ids = torch.tensor(
            [self.hero_id_to_idx[h["hero_id"]] for h in row["dire_heroes"][:5]],
            dtype=torch.long,
        )
        label = torch.tensor(1.0 if row["winning_team"] == "radiant" else 0.0, dtype=torch.float32)
        return radiant_ids, dire_ids, label


class HeroPickMLP(nn.Module):
    def __init__(self, num_heroes: int, embed_dim: int = 32, hidden_dim: int = 128, drop_rate: float = 0.3):
        super().__init__()
        self.hero_embedding = nn.Embedding(num_heroes, embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, radiant_ids, dire_ids):
        radiant_emb = self.hero_embedding(radiant_ids).mean(dim=1)
        dire_emb = self.hero_embedding(dire_ids).mean(dim=1)
        combined = torch.cat([radiant_emb, dire_emb], dim=1)
        return self.mlp(combined).squeeze(-1)


# Build vocab (dense 0..N-1 indices from real hero ids, which aren't contiguous)
all_hero_ids = sorted({
    h["hero_id"]
    for _, row in dataset.df.iterrows()
    for h in list(row["radiant_heroes"]) + list(row["dire_heroes"])
})
hero_id_to_idx = {hid: i for i, hid in enumerate(all_hero_ids)}
num_heroes = len(all_hero_ids)
print(f"num_heroes={num_heroes}")

train_df, test_df = train_test_split(
    dataset.df, test_size=0.2, random_state=42, stratify=dataset.df["winning_team"]
)

train_ds = HeroPickDataset(train_df, hero_id_to_idx)
test_ds = HeroPickDataset(test_df, hero_id_to_idx)
train_loader_mlp = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader_mlp = DataLoader(test_ds, batch_size=64, shuffle=False)


In [ ]:
# Train with early stopping on test accuracy, restoring the best checkpoint.
# Without early stopping, this overfits the same way the vision models did,
# train accuracy climbing into the 70s while test accuracy plateaus in the low 50s.
model_mlp = HeroPickMLP(num_heroes=num_heroes).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model_mlp.parameters(), lr=1e-3, weight_decay=1e-4)

best_test_acc, patience, patience_counter, best_state = 0, 3, 0, None

for epoch in range(20):
    model_mlp.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for r_ids, d_ids, yb in train_loader_mlp:
        r_ids, d_ids, yb = r_ids.to(device), d_ids.to(device), yb.to(device)
        logits = model_mlp(r_ids, d_ids)
        loss = loss_fn(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(yb)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        train_correct += (preds == yb).sum().item()
        train_total += len(yb)

    model_mlp.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for r_ids, d_ids, yb in test_loader_mlp:
            r_ids, d_ids, yb = r_ids.to(device), d_ids.to(device), yb.to(device)
            logits = model_mlp(r_ids, d_ids)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            test_correct += (preds == yb).sum().item()
            test_total += len(yb)
    test_acc = test_correct / test_total

    print(f"Epoch {epoch+1:02d} | train_loss={train_loss/train_total:.4f} "
          f"train_acc={train_correct/train_total:.4f} test_acc={test_acc:.4f}")

    if test_acc > best_test_acc:
        best_test_acc = test_acc
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model_mlp.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stop at epoch {epoch+1}")
            break

model_mlp.load_state_dict(best_state)
model_mlp.to(device)
print(f"\nRestored best model, test_acc={best_test_acc:.4f}")


## 8. Visualizing what the embedding MLP learned

Since the embedding table is built directly from hero_id, it's guaranteed to
encode "which hero is this" by construction. Worth seeing whether it organized
heroes into anything sensible: t-SNE of the learned vectors, nearest-neighbor
lookups, and a per-hero "swap-in" impact test.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

with open(Path(ICONS_DIR).parent / "metadata.json") as f:
    hero_metadata = json.load(f)

idx_to_hero_id = {v: k for k, v in hero_id_to_idx.items()}
hero_names = [hero_metadata[str(idx_to_hero_id[i])]["localized_name"] for i in range(num_heroes)]
print(f"Loaded {len(hero_names)} hero names.")


In [ ]:
# t-SNE of the learned hero embedding space
embeddings = model_mlp.hero_embedding.weight.detach().cpu().numpy()

tsne = TSNE(n_components=2, perplexity=15, random_state=42)
coords = tsne.fit_transform(embeddings)

plt.figure(figsize=(16, 12))
plt.scatter(coords[:, 0], coords[:, 1], s=25, alpha=0.7)
for i, name in enumerate(hero_names):
    plt.annotate(name, (coords[i, 0], coords[i, 1]), fontsize=8, alpha=0.85)
plt.title("Learned hero embedding space (t-SNE)")
plt.tight_layout()
plt.show()


In [ ]:
# Nearest neighbors + full similarity heatmap
import seaborn as sns

sim_matrix = cosine_similarity(embeddings)

def nearest_heroes(hero_name, k=5):
    idx = hero_names.index(hero_name)
    sims = sim_matrix[idx]
    top_idx = np.argsort(-sims)[1:k+1]
    return [(hero_names[i], round(float(sims[i]), 3)) for i in top_idx]

for name in ["Pudge", "Anti-Mage", "Crystal Maiden"]:
    if name in hero_names:
        print(f"{name}: {nearest_heroes(name)}")

plt.figure(figsize=(20, 18))
sns.heatmap(sim_matrix, xticklabels=hero_names, yticklabels=hero_names,
            cmap="coolwarm", center=0, cbar_kws={"label": "cosine similarity"})
plt.title("Hero embedding similarity matrix")
plt.tight_layout()
plt.show()


In [ ]:
# Per-hero win-probability impact: swap one hero into a fixed lineup and see
# how the predicted Radiant win probability shifts.
model_mlp.eval()
baseline_radiant_ids = all_hero_ids[:5]
baseline_dire_ids = all_hero_ids[5:10]

base_radiant = torch.tensor([[hero_id_to_idx[h] for h in baseline_radiant_ids]])
base_dire = torch.tensor([[hero_id_to_idx[h] for h in baseline_dire_ids]])

impacts = {}
with torch.no_grad():
    for hero_id in all_hero_ids:
        idx = hero_id_to_idx[hero_id]
        test_radiant = base_radiant.clone()
        test_radiant[0, 0] = idx
        logit = model_mlp(test_radiant.to(device), base_dire.to(device))
        impacts[hero_id] = torch.sigmoid(logit).item()

impact_df = pd.DataFrame([
    {"hero": hero_metadata[str(hid)]["localized_name"], "win_prob": p}
    for hid, p in impacts.items()
]).sort_values("win_prob", ascending=False)

plt.figure(figsize=(10, 22))
plt.barh(impact_df["hero"], impact_df["win_prob"])
plt.xlabel("Predicted Radiant win probability with this hero swapped in")
plt.axvline(0.5, color="gray", linestyle="--")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 9. Looking at what the vision models actually saw

Once the baselines were trustworthy, it's worth going back to the image models
purely to see what they'd been looking at, since accuracy wasn't going to improve
from here. Load a saved checkpoint from the sweep, then run saliency maps and a
feature-space t-SNE colored by outcome.

Finding: saliency was smeared diffusely across the *entire* canvas, including
empty padding around the hero icons, rather than sharply focused on the portraits.
That lines up with the accuracy story, the model never found a strong, localized,
hero-specific feature to grab onto in the first place.

In [ ]:
# Find runs that have saved weights on disk
import json
from pathlib import Path
import pandas as pd

available_runs = []
for run_dir in sorted(RUNS_ROOT.iterdir()):
    if not run_dir.is_dir():
        continue
    weights_path = run_dir / "model_weights.pt"
    config_path = run_dir / "config.json"
    results_path = run_dir / "results.json"
    if weights_path.exists() and config_path.exists():
        with open(config_path) as f:
            cfg = json.load(f)
        best_test_acc = None
        if results_path.exists():
            with open(results_path) as f:
                res = json.load(f)
            best_test_acc = max(res.get("test_acc", [0]))
        available_runs.append({
            "name": run_dir.name,
            "model_name": cfg.get("model_name"),
            "drop_rate": cfg.get("drop_rate"),
            "random_order": cfg.get("random_order"),
            "best_test_acc": best_test_acc,
        })

runs_df = pd.DataFrame(available_runs).sort_values("best_test_acc", ascending=False)
print(runs_df)


In [ ]:
# Load the top run's weights and rebuild matching loaders
run_name = runs_df.iloc[0]["name"]
run_dir = RUNS_ROOT / run_name

with open(run_dir / "config.json") as f:
    cfg_dict = json.load(f)

print(f"Loading run: {run_name}")
print(cfg_dict)

vit_model = HeroPicViT(
    model_name=cfg_dict["model_name"],
    pretrained=cfg_dict["pretrained"],
    drop_rate=cfg_dict["drop_rate"],
)
vit_model.load_state_dict(torch.load(run_dir / "model_weights.pt", map_location=device))
vit_model.to(device)
vit_model.eval()

data_config = resolve_data_config({}, model=vit_model.backbone)
CELL_H, CELL_W = 32, 32
vit_transforms = transforms.Compose([
    transforms.Resize((CELL_H, CELL_W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=data_config["mean"], std=data_config["std"]),
])

vit_train_loader, vit_val_loader, vit_test_loader, *_ = create_dataloaders(
    parquet_dir=str(PARQUET_PATH),
    icons_dir=str(ICONS_DIR),
    transform=vit_transforms,
    random_order=cfg_dict["random_order"],
    batch_size=8,
    val_split=0.15,
    test_split=0.15,
    num_workers=0,
)

print("Model + loaders ready for visualization.")


In [ ]:
# Saliency maps: gradient magnitude with respect to the input image
def denorm2(t, mean, std):
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t = torch.tensor(std).view(3, 1, 1)
    return (t.cpu() * std_t + mean_t).clamp(0, 1)

def show_saliency(model_, sample_tensor, label, data_config_, device_):
    model_.eval()
    x = sample_tensor.clone().unsqueeze(0).to(device_)
    x.requires_grad_(True)
    logit = model_(x).squeeze()
    logit.backward()
    saliency = x.grad.abs().squeeze(0).max(dim=0)[0].cpu().numpy()
    img = denorm2(sample_tensor, data_config_["mean"], data_config_["std"]).permute(1, 2, 0).numpy()

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(img); axes[0].set_title(f"Input (label={label})"); axes[0].axis("off")
    axes[1].imshow(img); axes[1].imshow(saliency, cmap="hot", alpha=0.6)
    axes[1].set_title("Saliency overlay"); axes[1].axis("off")
    plt.tight_layout()
    plt.show()

X, y = next(iter(vit_test_loader))
for i in range(4):
    show_saliency(vit_model, X[i], y[i].item(), data_config, device)


In [ ]:
# Feature-space t-SNE colored by outcome: if the image model had learned
# something about match outcome, this should show separation by color.
from sklearn.manifold import TSNE

vit_model.eval()
all_feats, all_labels = [], []
with torch.no_grad():
    for X, y in vit_test_loader:
        feats = vit_model.backbone(X.to(device))
        all_feats.append(feats.cpu().numpy())
        all_labels.append(y.numpy())

all_feats = np.concatenate(all_feats)
all_labels = np.concatenate(all_labels)

coords = TSNE(n_components=2, random_state=42).fit_transform(all_feats)
plt.figure(figsize=(10, 8))
sc = plt.scatter(coords[:, 0], coords[:, 1], c=all_labels, cmap="coolwarm", alpha=0.6)
plt.title(f"{run_name} feature space colored by match outcome")
plt.colorbar(sc, label="label (1=Radiant win)")
plt.tight_layout()
plt.show()


## 10. Where this leaves the scoreboard

| Approach | Test accuracy |
|---|---|
| Vision models (ViT Base/Small/Tiny, DeiT3, Swin, ResNet18, all variants) | ~49-53%, indistinguishable from chance |
| Gradient Boosted Trees on one-hot hero vectors | ~54.3-54.5%, AUC ~0.56 |
| Embedding-based MLP | ~54.5-55.5% at best epoch |

The draft does carry a small real signal, "we already lost in the draft" has a
kernel of truth to it, but it's a modest edge, not a dominant one. Turning that
signal into an image actively hides it rather than revealing anything extra
hiding in the pixels, regardless of which architecture, training regime, or hero
ordering was used.